## Randomly Sampling Images of Known HPMS

In this notebook, we seek to randpmly sample the images of our example HPMS for coordinates and see if KBMOD detects anything. The purpose of this is to gague the false positive rate of KBMOD on coadded images.

In [4]:
import logging
logging.basicConfig(level=logging.INFO)
from astropy.table import Table

In [5]:
filepath = "./kbmod_fits_example"

from kbmod.image_collection import ImageCollection
from kbmod.standardizers.fits_standardizers.kbmodv05 import KBMODV0_5, KBMODV0_5Config
from kbmod.configuration import SearchConfiguration
from kbmod.run_search import SearchRunner

ps1_bit_flag_map = {
    "DETECTOR": 2**0,
    "FLAT": 2**1,
    "DARK": 2**2,
    "BLANK": 2**3,
    "CTE": 2**4,
    "SAT": 2**5,
    "LOW": 2**6,
    "SUSPECT": 2**7,
    "BURNTOOL": 2**8,
    "CR": 2**9,
    "SPIKE": 2**10,
    "GHOST": 2**11,
    "STREAK": 2**12,
    "STARCORE": 2**13,
    "CONV.BAD": 2**14,
    "CONV.POOR": 2**15,
    "MARK": 2**16
}

ps1_mask_flags = ["DETECTOR", "BLANK", "CR", 
                  "SPIKE", "GHOST", "STARCORE",
                  "CONV.BAD", "STREAK", "BURNTOOL"]

load_config = KBMODV0_5Config(mask_flags=ps1_mask_flags, bit_flag_map=ps1_bit_flag_map)

# Loading from the files takes a while (multiple minutes).
ic = ImageCollection.fromDir(filepath, force=KBMODV0_5, config=load_config)
wu = ic.toWorkUnit()
wu.print_stats()

INFO:kbmod.image_collection:Creating ImageCollection from 29 standardizers.
INFO:kbmod.image_collection:Building WorkUnit from ImageCollection


WorkUnit:
  Num Constituent Images (29):
  Reprojected: False
Image Stack Statistics:
  Image Count: 29
  Image Size: 6295 x 6261 = 39412995
+------+------------+------------+------------+------------+----------+----------+----------+--------+
|  idx |     Time   |  Flux Min  |  Flux Max  |  Flux Mean |  Var Min |  Var Max | Var Mean | Masked |
+------+------------+------------+------------+------------+----------+----------+----------+--------+
|    0 |  56793.445 |    -148.94 |   23410.41 |       1.04 |     0.00 |     0.01 |     0.00 |  26.07 |
+------+------------+------------+------------+------------+----------+----------+----------+--------+
|    1 |  56759.553 |    -152.22 |   30344.73 |       1.05 |     0.00 |     0.01 |     0.01 |  28.21 |
+------+------------+------------+------------+------------+----------+----------+----------+--------+
|    2 |  55340.340 |    -174.64 |   13124.54 |       3.99 |     0.00 |     0.01 |     0.00 |  62.08 |
+------+------------+------------+-

In [ ]:
wu.im_stack.mask_by_science_bounds(min_val=0.0)
# wu.im_stack.mask_by_variance_bounds(min_val=1e-10)

wu.print_stats()

In [ ]:
import random

def new_sample_config(vx_min, vx_max, vy_min, vy_max,
                      v_steps, x_pixels, y_pixels, x_width,
                      y_width, v_width):
    
    new_x = random.randint(0, x_pixels)
    new_y = random.randint(0, y_pixels)

    
    input_parameters = {
    "x_pixel_bounds": [new_x - x_width/2 , new_x + x_width/2],
    "y_pixel_bounds": [new_y - y_width/2, new_y + y_width/2],

    # Use search parameters (including a force ecliptic angle of 0.0)
    # to match what we know is in the demo data.
    "generator_config": {
        "name": "VelocityGridSearch",
        "vx_steps": v_steps,
        "min_vx": vx_min - v_width/2,
        "max_vx": vx_max + v_width/2,
        "vy_steps": v_steps,
        "min_vy": vx_min - v_width/2,
        "max_vy": vx_min + v_width/2,
    },
    # Output parameters
    "result_filename": "./results.ecsv",
    # Basic filtering (always applied)
    "num_obs": 15,  # <-- Filter anything with fewer than 15 observations
    "lh_level": 10.0,  # <-- Filter anything with a likelihood < 10.0
    # SigmaG clipping parameters
    "sigmaG_lims": [25, 75],  # <-- Clipping parameters (lower and upper percentile)
    # Other parameters
    "cpu_only": True,  # <-- This will be absurdly slow.  Set to False if you have a good enough GPU.
    "coadds": ["mean","median"],
    "save_all_stamps": True
    }
    config = SearchConfiguration.from_dict(input_parameters)

    # Make the WorkUnit use this configuration.
    return config

def sample_new_coords(vx_min, vx_max, vy_min, vy_max,
                      v_steps, x_pixels, y_pixels, x_width,
                      y_width, v_width, wu):
    
    wu.config = new_sample_config(vx_min, vx_max, vy_min, vy_max,
                                  v_steps, x_pixels, y_pixels, x_width,
                                  y_width, v_width)
    
    rs = SearchRunner()
    results = rs.run_search_from_work_unit(wu)
    table = Table.read("results.ecsv", format='ascii.ecsv')
    return len(table)

In [ ]:
vx_min=
vx_max=
vy_min=
vy_max=
v_steps=
x_pixels=
y_pixels=
x_width=
y_width=
v_width=

In [ ]:
results = [sample_new_coords(vx_min, vx_max, vy_min, vy_max, v_steps, x_pixels, y_pixels, x_width, y_width, v_width, wu) for i in range(100)]